# 2. Data Preprocessing

In this section, the raw data obtained in the previous step has been made suitable for analysis and modeling.
Missing values, data types, and unnecessary columns have been addressed at this stage.

### Labeling Strategy (Binary Sentiment)

The original rating scale contains neutral reviews (around 3–4).  
To keep the problem as a clear binary classification task (positive vs. negative) and reduce label noise,  
neutral samples were excluded from training and evaluation.


In [4]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

df = pd.read_csv("raw_reviews.csv", encoding="utf-8-sig")

def preprocess_data(df):
    def label_sentiment(puan_str):
        try:
            puan = float(puan_str.replace(',', '.'))
            if puan >= 4.0: return 1  # Positive
            elif puan <= 3: return 0 # Negative
            else: return None
        except:
            return None

    df['label'] = df['Puan'].apply(label_sentiment)
    df_filtered = df.dropna(subset=['label']).copy()
  

    stop_words = set(stopwords.words('turkish'))
    lemmatizer = WordNetLemmatizer()

    def clean_text(text):
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r'<.*?>', '', text)
        text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
        text = re.sub(r'\d+', '', text)
        # Tokenizasyon 
        tokens = nltk.word_tokenize(text)
        # Stop words kaldırma ve Lemmatization 
        cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
        return " ".join(cleaned)


    df_filtered['clean_text'] = df_filtered['Yorum'].apply(clean_text)
    return df_filtered


df_final = preprocess_data(df)

print(f"Toplam İşlenen Veri Sayısı: {len(df_final)}")
print(f"Sınıf Dağılımı:\n{df_final['label'].value_counts()}")
print(f"Sınıf Dağılımı (Oran):\n{df_final['label'].value_counts(normalize=True)}")

# Kaydet
df_final.to_csv("processed_reviews.csv", index=False, encoding="utf-8-sig")
print(f"Kaydedildi processed_reviews.csv | Satır sayısı: {len(df_final)}")


Toplam İşlenen Veri Sayısı: 2094
Sınıf Dağılımı:
label
1.0    1212
0.0     882
Name: count, dtype: int64
Sınıf Dağılımı (Oran):
label
1.0    0.578797
0.0    0.421203
Name: proportion, dtype: float64
Kaydedildi processed_reviews.csv | Satır sayısı: 2094


In [5]:
df_final.describe()

,label
count,2094.000000
mean,0.578797
std,0.493870
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000
